# Small CNN 4,996 Parameters MNIST Training

Run these cells top-to-bottom in Google Colab to train the float model, fine-tune with QAT, convert to INT8, then report accuracy, inference time, and model size.


In [ ]:
import copy
import os
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.ao.quantization import (
    DeQuantStub,
    QuantStub,
    convert,
    get_default_qat_qconfig,
    prepare_qat,
)
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


In [ ]:
# Colab-friendly configuration. Edit these values before retraining.
SEED = 1
DATA_DIR = "./data"
BATCH_SIZE = 64
TEST_BATCH_SIZE = 64
FLOAT_EPOCHS = 14
QAT_EPOCHS = 10
FLOAT_LR = 0.01
QAT_LR = 1e-4
MOMENTUM = 0.9
LOG_INTERVAL = 100
AUGMENT = False
NORMALIZE = False
QCONFIG_BACKEND = "fbgemm"
BENCHMARK_BATCH_SIZE = 1
BENCHMARK_REPEATS = 1000
SAVE_CHECKPOINTS = True
FLOAT_CHECKPOINT = "small_cnn_float_state_dict.pt"
INT8_CHECKPOINT = "small_cnn_int8_state_dict.pt"
INT8_WEIGHTS_TXT = "small_cnn_int8_weights_c_order.txt"
INT8_BIASES_TXT = "small_cnn_biases_float_c_order.txt"
INT8_BIASES_INT32_TXT = "small_cnn_biases_int32_c_order.txt"
INT8_QPARAMS_TXT = "small_cnn_int8_quant_params.txt"
INT8_EXPORT_LAYOUT = "c"  # "c" matches the existing C model layout; "pytorch" keeps module layout.
# Signed symmetric INT8 artifacts for RTL golden inference.
SYM_WEIGHTS_TXT = "small_cnn_sym_weights_i8_c_order.txt"
SYM_BIASES_INT32_TXT = "small_cnn_sym_biases_i32_c_order.txt"
SYM_REQUANT_MULT_TXT = "small_cnn_sym_requant_mult_i32_c_order.txt"
SYM_REQUANT_SHIFT_TXT = "small_cnn_sym_requant_shift_u6_c_order.txt"
SYM_QPARAMS_TXT = "small_cnn_sym_qparams.txt"
SYM_REFERENCE_PY = "reference_infer_int.py"
SYM_REQUANT_SHIFT = 31
SYM_CALIBRATION_BATCHES = None  # None uses the full test loader for activation scales.
SYM_ACCURACY_MAX_IMAGES = None  # None evaluates the full test dataset with integer inference.
SYM_DEBUG_HEX_FILES = {
    "input": "input_image_i8.hex",
    "layer0_acc": "layer0_conv_acc_i32.hex",
    "layer0_out": "layer0_out_i8.hex",
    "layer1_acc": "layer1_conv_acc_i32.hex",
    "layer1_out": "layer1_out_i8.hex",
    "layer2_acc": "layer2_fc_acc_i32.hex",
    "layer2_out": "layer2_out_i8.hex",
    "layer3_acc": "layer3_fc_acc_i32.hex",
    "final_logits": "final_logits_i8.hex",
}
INFERENCE_SAMPLE_INDEX = 0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

print(f"Using device: {DEVICE}")


In [ ]:
class CNN(nn.Module):
    """Small MNIST CNN with 4,996 trainable parameters."""

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 8, kernel_size=3)  # 80 params, output: 8x26x26
        self.conv2 = nn.Conv2d(8, 10, kernel_size=3)  # 730 params, output: 10x11x11
        self.mp = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(10 * 5 * 5, 16)  # 4,016 params
        self.fc2 = nn.Linear(16, 10)  # 170 params

        self.quant = QuantStub()
        self.dequant = DeQuantStub()

    def forward(self, x):
        x = self.quant(x)
        x = self.mp(F.relu(self.conv1(x)))  # 8x13x13
        x = self.mp(F.relu(self.conv2(x)))  # 10x5x5
        x = x.reshape(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        x = self.dequant(x)
        return F.log_softmax(x, dim=1)


In [ ]:
def build_transform(augment=False, normalize=False):
    transform_steps = []
    if augment:
        transform_steps.append(transforms.RandomAffine(degrees=15, translate=(0.1, 0.1)))
    transform_steps.append(transforms.ToTensor())
    if normalize:
        transform_steps.append(transforms.Normalize((0.1307,), (0.3081,)))
    return transforms.Compose(transform_steps)


def build_loaders(data_dir, batch_size, test_batch_size, augment=False, normalize=False, download=True):
    train_dataset = datasets.MNIST(
        root=str(data_dir),
        train=True,
        transform=build_transform(augment=augment, normalize=normalize),
        download=download,
    )
    test_dataset = datasets.MNIST(
        root=str(data_dir),
        train=False,
        transform=build_transform(augment=False, normalize=normalize),
        download=download,
    )

    loader_kwargs = {
        "num_workers": 2,
        "pin_memory": DEVICE.type == "cuda",
    }
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, **loader_kwargs)
    test_loader = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=False, **loader_kwargs)
    return train_loader, test_loader


train_loader, test_loader = build_loaders(
    data_dir=DATA_DIR,
    batch_size=BATCH_SIZE,
    test_batch_size=TEST_BATCH_SIZE,
    augment=AUGMENT,
    normalize=NORMALIZE,
    download=True,
)

print(f"Train samples: {len(train_loader.dataset)}")
print(f"Test samples: {len(test_loader.dataset)}")


In [ ]:
def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters())


def model_size_kb(model, filename="temp.p"):
    path = Path(filename)
    torch.save(model.state_dict(), path)
    try:
        return os.path.getsize(path) / 1024
    finally:
        path.unlink(missing_ok=True)


def sync_if_needed(device):
    if device.type == "cuda":
        torch.cuda.synchronize()


def benchmark_inference(model, device, batch_size=1, warmup=100, repeats=1000):
    model.eval()
    sample = torch.randn(batch_size, 1, 28, 28, device=device)

    with torch.inference_mode():
        for _ in range(warmup):
            model(sample)

        sync_if_needed(device)
        start = time.perf_counter()
        for _ in range(repeats):
            model(sample)
        sync_if_needed(device)
        elapsed_s = time.perf_counter() - start

    images = batch_size * repeats
    ms_per_image = elapsed_s * 1000 / images
    images_per_second = images / elapsed_s
    return ms_per_image, images_per_second


In [ ]:
def train_epoch(model, train_loader, optimizer, epoch, device, log_interval):
    model.train()
    running_loss = 0.0

    for batch_idx, (data, target) in enumerate(train_loader):
        data = data.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * data.size(0)
        if log_interval > 0 and batch_idx % log_interval == 0:
            processed = batch_idx * len(data)
            total = len(train_loader.dataset)
            percent = 100.0 * batch_idx / len(train_loader)
            print(
                f"Train Epoch: {epoch} [{processed}/{total} ({percent:.0f}%)]"
                f" Loss: {loss.item():.6f}"
            )

    return running_loss / len(train_loader.dataset)


def evaluate(model, test_loader, device):
    model.eval()
    test_loss = 0.0
    correct = 0

    with torch.inference_mode():
        for data, target in test_loader:
            data = data.to(device, non_blocking=True)
            target = target.to(device, non_blocking=True)
            output = model(data)

            test_loss += F.nll_loss(output, target, reduction="sum").item()
            pred = output.argmax(dim=1)
            correct += pred.eq(target).sum().item()

    test_loss /= len(test_loader.dataset)
    accuracy = 100.0 * correct / len(test_loader.dataset)
    print()
    print(
        f"Test set: Average loss: {test_loss:.4f}, "
        f"Accuracy: {correct}/{len(test_loader.dataset)} ({accuracy:.2f}%)"
    )
    print()
    return test_loss, correct, accuracy


In [ ]:
def train_float_model(model, train_loader, test_loader, device, epochs, lr, momentum, log_interval):
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

    for epoch in range(1, epochs + 1):
        train_epoch(model, train_loader, optimizer, epoch, device, log_interval)
        scheduler.step()
        evaluate(model, test_loader, device)

    return model


def train_qat_model(model, train_loader, test_loader, device, epochs, lr, momentum, log_interval, qconfig_backend):
    if qconfig_backend in torch.backends.quantized.supported_engines:
        torch.backends.quantized.engine = qconfig_backend

    qat_model = copy.deepcopy(model)
    qat_model.qconfig = get_default_qat_qconfig(qconfig_backend)
    qat_model.train()
    prepared_model = prepare_qat(qat_model).to(device)
    optimizer = optim.SGD(prepared_model.parameters(), lr=lr, momentum=momentum)

    for epoch in range(1, epochs + 1):
        train_epoch(prepared_model, train_loader, optimizer, epoch, device, log_interval)
        evaluate(prepared_model, test_loader, device)

    return prepared_model


In [ ]:
model = CNN().to(DEVICE)
total_params = count_parameters(model)
print(model)
print(f"Number of parameters: {total_params}")
assert total_params == 4996, f"Expected 4,996 parameters, got {total_params}"


In [ ]:
model = train_float_model(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    device=DEVICE,
    epochs=FLOAT_EPOCHS,
    lr=FLOAT_LR,
    momentum=MOMENTUM,
    log_interval=LOG_INTERVAL,
)


In [ ]:
_float_loss, float_correct, float_accuracy = evaluate(model, test_loader, DEVICE)
float_ms, float_ips = benchmark_inference(
    model,
    device=DEVICE,
    batch_size=BENCHMARK_BATCH_SIZE,
    repeats=BENCHMARK_REPEATS,
)

print(f"Float accuracy: {float_correct}/{len(test_loader.dataset)} ({float_accuracy:.2f}%)")
print(f"Float inference: {float_ms:.6f} ms/image, {float_ips:.2f} images/s")
print(f"Float model size: {model_size_kb(model, 'float.p'):.3f} KB")

if SAVE_CHECKPOINTS:
    torch.save(model.state_dict(), FLOAT_CHECKPOINT)
    print(f"Saved float checkpoint: {FLOAT_CHECKPOINT}")


In [ ]:
prepared_model = train_qat_model(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    device=DEVICE,
    epochs=QAT_EPOCHS,
    lr=QAT_LR,
    momentum=MOMENTUM,
    log_interval=LOG_INTERVAL,
    qconfig_backend=QCONFIG_BACKEND,
)


In [ ]:
_qat_loss, qat_correct, qat_accuracy = evaluate(prepared_model, test_loader, DEVICE)
print(f"QAT accuracy before convert: {qat_correct}/{len(test_loader.dataset)} ({qat_accuracy:.2f}%)")


In [ ]:
model_int8 = convert(prepared_model.to("cpu").eval())
int8_device = torch.device("cpu")
_int8_loss, int8_correct, int8_accuracy = evaluate(model_int8, test_loader, int8_device)
int8_ms, int8_ips = benchmark_inference(
    model_int8,
    device=int8_device,
    batch_size=BENCHMARK_BATCH_SIZE,
    repeats=BENCHMARK_REPEATS,
)

print(f"PyTorch affine INT8 accuracy: {int8_correct}/{len(test_loader.dataset)} ({int8_accuracy:.2f}%)")
print(f"PyTorch affine INT8 inference: {int8_ms:.6f} ms/image, {int8_ips:.2f} images/s")
print(f"PyTorch affine INT8 model size: {model_size_kb(model_int8, 'int8.p'):.3f} KB")

if SAVE_CHECKPOINTS:
    torch.save(model_int8.state_dict(), INT8_CHECKPOINT)
    print(f"Saved PyTorch affine INT8 checkpoint: {INT8_CHECKPOINT}")


In [ ]:
SYMMETRIC_LAYER_SPECS = [
    {"name": "conv1", "kind": "conv", "input_scale": "input", "output_scale": "conv1"},
    {"name": "conv2", "kind": "conv", "input_scale": "conv1", "output_scale": "conv2"},
    {"name": "fc1", "kind": "linear", "input_scale": "conv2", "output_scale": "fc1"},
    {"name": "fc2", "kind": "linear", "input_scale": "fc1", "output_scale": "fc2"},
]

WEIGHT_BASE_OFFSETS = {"conv1": 0, "conv2": 72, "fc1": 792, "fc2": 4792}
BIAS_BASE_OFFSETS = {"conv1": 0, "conv2": 8, "fc1": 18, "fc2": 34}
EXPECTED_SYM_WEIGHT_COUNT = 4952
EXPECTED_SYM_BIAS_COUNT = 44
if "SYM_DEBUG_HEX_FILES" not in globals():
    SYM_DEBUG_HEX_FILES = {
        "input": "input_image_i8.hex",
        "layer0_acc": "layer0_conv_acc_i32.hex",
        "layer0_out": "layer0_out_i8.hex",
        "layer1_acc": "layer1_conv_acc_i32.hex",
        "layer1_out": "layer1_out_i8.hex",
        "layer2_acc": "layer2_fc_acc_i32.hex",
        "layer2_out": "layer2_out_i8.hex",
        "layer3_acc": "layer3_fc_acc_i32.hex",
        "final_logits": "final_logits_i8.hex",
    }


def _symmetric_scale_from_max(max_abs):
    return max(float(max_abs), 1e-12) / 127.0


def _bias_tensor(module, output_count):
    bias = getattr(module, "bias", None)
    if bias is None:
        return torch.zeros(output_count, dtype=torch.float64)
    return bias.detach().cpu().to(torch.float64).reshape(-1)


def _quantize_weight_symmetric_per_output(module):
    weight = module.weight.detach().cpu().to(torch.float64)
    output_count = weight.shape[0]
    flat = weight.reshape(output_count, -1)
    max_abs = flat.abs().amax(dim=1)
    scales = torch.clamp(max_abs, min=1e-12) / 127.0
    view_shape = [output_count] + [1] * (weight.dim() - 1)
    qweight = torch.round(weight / scales.reshape(view_shape)).clamp(-128, 127).to(torch.int8)
    return qweight.contiguous(), scales.tolist()


def _flatten_symmetric_weight_for_export(layer_name, qweight):
    if layer_name.startswith("fc"):
        return qweight.t().contiguous().reshape(-1)
    return qweight.contiguous().reshape(-1)


def collect_symmetric_activation_scales(model, data_loader, max_batches=None):
    export_model = copy.deepcopy(model).to("cpu").eval()
    max_abs = {"input": 0.0, "conv1": 0.0, "conv2": 0.0, "fc1": 0.0, "fc2": 0.0}

    with torch.inference_mode():
        for batch_index, (images, _labels) in enumerate(data_loader):
            if max_batches is not None and batch_index >= max_batches:
                break

            x = images.to("cpu")
            max_abs["input"] = max(max_abs["input"], float(x.abs().max().item()))

            conv1 = F.relu(export_model.conv1(x))
            max_abs["conv1"] = max(max_abs["conv1"], float(conv1.abs().max().item()))
            x = export_model.mp(conv1)

            conv2 = F.relu(export_model.conv2(x))
            max_abs["conv2"] = max(max_abs["conv2"], float(conv2.abs().max().item()))
            x = export_model.mp(conv2)

            x = x.reshape(x.size(0), -1)
            fc1 = F.relu(export_model.fc1(x))
            max_abs["fc1"] = max(max_abs["fc1"], float(fc1.abs().max().item()))

            fc2 = export_model.fc2(fc1)
            max_abs["fc2"] = max(max_abs["fc2"], float(fc2.abs().max().item()))

    scales = {name: _symmetric_scale_from_max(value) for name, value in max_abs.items()}
    return scales, max_abs


def _make_requant_params(input_scale, output_scale, weight_scales, shift):
    multipliers = []
    requant_scales = []
    for weight_scale in weight_scales:
        requant_scale = float(input_scale) * float(weight_scale) / float(output_scale)
        multiplier = int(round(requant_scale * (1 << shift)))
        if multiplier < -(1 << 31) or multiplier > (1 << 31) - 1:
            raise ValueError(f"Requant multiplier {multiplier} does not fit signed int32")
        requant_scales.append(requant_scale)
        multipliers.append(multiplier)
    shifts = [int(shift)] * len(multipliers)
    return requant_scales, multipliers, shifts


def _format_float_list(values):
    return " ".join(f"{float(value):.12g}" for value in values)


def _format_int_list(values):
    return " ".join(str(int(value)) for value in values)


def export_symmetric_int8_artifacts(
    model,
    data_loader,
    weights_path=SYM_WEIGHTS_TXT,
    biases_path=SYM_BIASES_INT32_TXT,
    requant_mult_path=SYM_REQUANT_MULT_TXT,
    requant_shift_path=SYM_REQUANT_SHIFT_TXT,
    qparams_path=SYM_QPARAMS_TXT,
    calibration_batches=SYM_CALIBRATION_BATCHES,
    requant_shift=SYM_REQUANT_SHIFT,
):
    export_model = copy.deepcopy(model).to("cpu").eval()
    activation_scales, activation_max_abs = collect_symmetric_activation_scales(
        export_model,
        data_loader,
        max_batches=calibration_batches,
    )

    all_weights = []
    all_biases = []
    all_requant_multipliers = []
    all_requant_shifts = []
    layer_states = {}
    qparam_lines = [
        "# Signed symmetric INT8 export for small_cnn_4996_params",
        "# Numeric contract: int8 activations, int8 weights, int32 accumulators, int32 biases.",
        "zero_point_policy=all_zero",
        "activation_dtype=int8",
        "weight_dtype=int8",
        "bias_dtype=int32",
        "accumulator_dtype=int32",
        f"requant_shift={int(requant_shift)}",
        "rounding=rtl_round_shift",
        "weight_export_order_conv=W[oc][ic][ky][kx]",
        "weight_export_order_fc=W[input_index][output_channel]",
        "",
    ]

    for name in ["input", "conv1", "conv2", "fc1", "fc2"]:
        qparam_lines.extend([
            f"activation_scale.{name}={activation_scales[name]:.12g}",
            f"activation_zero_point.{name}=0",
            f"activation_max_abs.{name}={activation_max_abs[name]:.12g}",
        ])
    qparam_lines.append("")

    for spec in SYMMETRIC_LAYER_SPECS:
        layer_name = spec["name"]
        module = getattr(export_model, layer_name)
        weight_q, weight_scales = _quantize_weight_symmetric_per_output(module)
        output_count = int(weight_q.shape[0])
        bias_float = _bias_tensor(module, output_count)
        input_scale = activation_scales[spec["input_scale"]]
        output_scale = activation_scales[spec["output_scale"]]

        if len(all_weights) != WEIGHT_BASE_OFFSETS[layer_name]:
            raise RuntimeError(f"Unexpected weight offset before {layer_name}: {len(all_weights)}")
        if len(all_biases) != BIAS_BASE_OFFSETS[layer_name]:
            raise RuntimeError(f"Unexpected bias offset before {layer_name}: {len(all_biases)}")

        bias_scales = [input_scale * float(weight_scale) for weight_scale in weight_scales]
        bias_q = [
            int(round(float(bias_value) / float(bias_scale))) if bias_scale != 0.0 else 0
            for bias_value, bias_scale in zip(bias_float.tolist(), bias_scales)
        ]
        for value in bias_q:
            if value < -(1 << 31) or value > (1 << 31) - 1:
                raise ValueError(f"Bias value {value} does not fit signed int32")

        requant_scales, requant_multipliers, requant_shifts = _make_requant_params(
            input_scale=input_scale,
            output_scale=output_scale,
            weight_scales=weight_scales,
            shift=requant_shift,
        )
        for value in requant_shifts:
            if value < 0 or value > 63:
                raise ValueError(f"Requant shift {value} does not fit unsigned 6-bit")

        flat_weight = _flatten_symmetric_weight_for_export(layer_name, weight_q)
        all_weights.extend(int(value) for value in flat_weight.tolist())
        all_biases.extend(int(value) for value in bias_q)
        all_requant_multipliers.extend(int(value) for value in requant_multipliers)
        all_requant_shifts.extend(int(value) for value in requant_shifts)

        layer_states[layer_name] = {
            "kind": spec["kind"],
            "weight_q": weight_q.numpy().astype(np.int8),
            "bias_q": np.asarray(bias_q, dtype=np.int32),
            "requant_multiplier": np.asarray(requant_multipliers, dtype=np.int32),
            "requant_shift": np.asarray(requant_shifts, dtype=np.int64),
            "input_scale": float(input_scale),
            "output_scale": float(output_scale),
            "weight_scales": np.asarray(weight_scales, dtype=np.float64),
        }

        qparam_lines.extend([
            f"[{layer_name}]",
            f"kind={spec['kind']}",
            f"weight_base={WEIGHT_BASE_OFFSETS[layer_name]}",
            f"bias_base={BIAS_BASE_OFFSETS[layer_name]}",
            f"weight_shape={list(weight_q.shape)}",
            f"exported_weight_count={int(flat_weight.numel())}",
            f"bias_count={len(bias_q)}",
            f"input_scale_name={spec['input_scale']}",
            f"output_scale_name={spec['output_scale']}",
            f"input_scale={input_scale:.12g}",
            "input_zero_point=0",
            f"output_scale={output_scale:.12g}",
            "output_zero_point=0",
            "weight_zero_points=" + _format_int_list([0] * len(weight_scales)),
            "weight_scales=" + _format_float_list(weight_scales),
            "bias_scales=" + _format_float_list(bias_scales),
            "requant_scales=" + _format_float_list(requant_scales),
            "requant_multipliers=" + _format_int_list(requant_multipliers),
            "requant_shifts=" + _format_int_list(requant_shifts),
            "",
        ])

    if len(all_weights) != EXPECTED_SYM_WEIGHT_COUNT:
        raise RuntimeError(f"Expected {EXPECTED_SYM_WEIGHT_COUNT} weights, exported {len(all_weights)}")
    if len(all_biases) != EXPECTED_SYM_BIAS_COUNT:
        raise RuntimeError(f"Expected {EXPECTED_SYM_BIAS_COUNT} biases, exported {len(all_biases)}")
    if len(all_requant_multipliers) != EXPECTED_SYM_BIAS_COUNT:
        raise RuntimeError("Requant multiplier count does not match output-channel count")

    output_paths = [
        Path(weights_path),
        Path(biases_path),
        Path(requant_mult_path),
        Path(requant_shift_path),
        Path(qparams_path),
    ]
    for output_path in output_paths:
        output_path.parent.mkdir(parents=True, exist_ok=True)

    Path(weights_path).write_text("".join(f"{value}\n" for value in all_weights))
    Path(biases_path).write_text("".join(f"{value}\n" for value in all_biases))
    Path(requant_mult_path).write_text("".join(f"{value}\n" for value in all_requant_multipliers))
    Path(requant_shift_path).write_text("".join(f"{value}\n" for value in all_requant_shifts))
    qparam_lines.extend([
        f"total_weight_count={len(all_weights)}",
        f"total_bias_count={len(all_biases)}",
        f"total_requant_multiplier_count={len(all_requant_multipliers)}",
        f"total_requant_shift_count={len(all_requant_shifts)}",
    ])
    Path(qparams_path).write_text("\n".join(qparam_lines) + "\n")

    export_info = {
        "weights_path": str(weights_path),
        "biases_path": str(biases_path),
        "requant_mult_path": str(requant_mult_path),
        "requant_shift_path": str(requant_shift_path),
        "qparams_path": str(qparams_path),
        "weight_count": len(all_weights),
        "bias_count": len(all_biases),
        "requant_count": len(all_requant_multipliers),
        "activation_scales": activation_scales,
    }
    export_state = {
        "layers": layer_states,
        "activation_scales": activation_scales,
        "debug_files": SYM_DEBUG_HEX_FILES,
    }
    print(f"Exported {len(all_weights)} signed symmetric INT8 weights to {weights_path}")
    print(f"Exported {len(all_biases)} signed INT32 biases to {biases_path}")
    print(f"Exported {len(all_requant_multipliers)} INT32 requant multipliers to {requant_mult_path}")
    print(f"Exported {len(all_requant_shifts)} U6 requant shifts to {requant_shift_path}")
    print(f"Exported symmetric qparams to {qparams_path}")
    return export_state, export_info


sym_export_state, sym_export_info = export_symmetric_int8_artifacts(prepared_model, test_loader)


In [ ]:
REFERENCE_INFER_INT_SOURCE = r'''
from pathlib import Path
import argparse
import numpy as np

WEIGHTS_FILE = "small_cnn_sym_weights_i8_c_order.txt"
BIASES_FILE = "small_cnn_sym_biases_i32_c_order.txt"
REQUANT_MULT_FILE = "small_cnn_sym_requant_mult_i32_c_order.txt"
REQUANT_SHIFT_FILE = "small_cnn_sym_requant_shift_u6_c_order.txt"
QPARAMS_FILE = "small_cnn_sym_qparams.txt"

WEIGHT_BASES = {"conv1": 0, "conv2": 72, "fc1": 792, "fc2": 4792}
BIAS_BASES = {"conv1": 0, "conv2": 8, "fc1": 18, "fc2": 34}
DEBUG_FILES = {
    "input": "input_image_i8.hex",
    "layer0_acc": "layer0_conv_acc_i32.hex",
    "layer0_out": "layer0_out_i8.hex",
    "layer1_acc": "layer1_conv_acc_i32.hex",
    "layer1_out": "layer1_out_i8.hex",
    "layer2_acc": "layer2_fc_acc_i32.hex",
    "layer2_out": "layer2_out_i8.hex",
    "layer3_acc": "layer3_fc_acc_i32.hex",
    "final_logits": "final_logits_i8.hex",
}


def read_int_file(path):
    return [int(line.strip()) for line in Path(path).read_text().splitlines() if line.strip()]


def hex_to_signed(token, bits):
    value = int(token.strip(), 16)
    sign_bit = 1 << (bits - 1)
    if value & sign_bit:
        value -= 1 << bits
    return value


def read_hex_file(path, bits):
    return [hex_to_signed(line, bits) for line in Path(path).read_text().splitlines() if line.strip()]


def signed_to_hex(value, bits):
    return f"{int(value) & ((1 << bits) - 1):0{bits // 4}x}"


def write_hex_file(path, values, bits):
    flat = np.asarray(values).reshape(-1)
    Path(path).write_text("".join(f"{signed_to_hex(value, bits)}\n" for value in flat))


def load_qparams(export_dir):
    qparams = {}
    path = Path(export_dir) / QPARAMS_FILE
    if not path.exists():
        return qparams
    for raw_line in path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        qparams[key.strip()] = value.strip()
    return qparams


def symmetric_quantize_float(array, scale):
    q = np.rint(np.asarray(array, dtype=np.float64) / float(scale))
    return np.clip(q, -128, 127).astype(np.int8)


def load_exported_state(export_dir):
    export_dir = Path(export_dir)
    weights = read_int_file(export_dir / WEIGHTS_FILE)
    biases = read_int_file(export_dir / BIASES_FILE)
    multipliers = read_int_file(export_dir / REQUANT_MULT_FILE)
    shifts = read_int_file(export_dir / REQUANT_SHIFT_FILE)

    if len(weights) != 4952:
        raise ValueError(f"Expected 4952 weights, got {len(weights)}")
    if len(biases) != 44 or len(multipliers) != 44 or len(shifts) != 44:
        raise ValueError("Expected 44 biases, 44 multipliers, and 44 shifts")

    conv1_w = np.asarray(weights[0:72], dtype=np.int8).reshape(8, 1, 3, 3)
    conv2_w = np.asarray(weights[72:792], dtype=np.int8).reshape(10, 8, 3, 3)
    fc1_w = np.asarray(weights[792:4792], dtype=np.int8).reshape(250, 16).T.copy()
    fc2_w = np.asarray(weights[4792:4952], dtype=np.int8).reshape(16, 10).T.copy()

    return {
        "conv1": {
            "weight_q": conv1_w,
            "bias_q": np.asarray(biases[0:8], dtype=np.int32),
            "requant_multiplier": np.asarray(multipliers[0:8], dtype=np.int32),
            "requant_shift": np.asarray(shifts[0:8], dtype=np.int64),
        },
        "conv2": {
            "weight_q": conv2_w,
            "bias_q": np.asarray(biases[8:18], dtype=np.int32),
            "requant_multiplier": np.asarray(multipliers[8:18], dtype=np.int32),
            "requant_shift": np.asarray(shifts[8:18], dtype=np.int64),
        },
        "fc1": {
            "weight_q": fc1_w,
            "bias_q": np.asarray(biases[18:34], dtype=np.int32),
            "requant_multiplier": np.asarray(multipliers[18:34], dtype=np.int32),
            "requant_shift": np.asarray(shifts[18:34], dtype=np.int64),
        },
        "fc2": {
            "weight_q": fc2_w,
            "bias_q": np.asarray(biases[34:44], dtype=np.int32),
            "requant_multiplier": np.asarray(multipliers[34:44], dtype=np.int32),
            "requant_shift": np.asarray(shifts[34:44], dtype=np.int64),
        },
    }


def rtl_round_shift_array(values, shift):
    shift = int(np.asarray(shift).reshape(-1)[0])
    values = np.asarray(values, dtype=np.int64)
    if shift == 0:
        return values
    offset = np.int64(1 << (shift - 1))
    return np.where(values >= 0, (values + offset) >> shift, (values - offset) >> shift)


def conv2d_acc_i32(input_i8, weight_i8):
    x = np.asarray(input_i8, dtype=np.int32)
    w = np.asarray(weight_i8, dtype=np.int32)
    kh, kw = w.shape[2], w.shape[3]
    windows = np.lib.stride_tricks.sliding_window_view(x, (kh, kw), axis=(1, 2))
    return np.tensordot(w, windows, axes=([1, 2, 3], [0, 3, 4])).astype(np.int32)


def fc_acc_i32(input_i8, weight_i8):
    return (np.asarray(weight_i8, dtype=np.int32) @ np.asarray(input_i8, dtype=np.int32)).astype(np.int32)


def requantize_i8(acc_i32, bias_i32, multiplier_i32, shift_u6, relu=False):
    acc = np.asarray(acc_i32, dtype=np.int64)
    bias = np.asarray(bias_i32, dtype=np.int64).reshape((-1,) + (1,) * (acc.ndim - 1))
    mult = np.asarray(multiplier_i32, dtype=np.int64).reshape((-1,) + (1,) * (acc.ndim - 1))
    tmp = acc + bias
    scaled = rtl_round_shift_array(tmp * mult, shift_u6)
    clipped = np.clip(scaled, -128, 127)
    if relu:
        clipped = np.maximum(clipped, 0)
    return clipped.astype(np.int8)


def maxpool2d_i8(input_i8):
    x = np.asarray(input_i8, dtype=np.int8)
    channels, height, width = x.shape
    out_h, out_w = height // 2, width // 2
    x = x[:, : out_h * 2, : out_w * 2]
    return x.reshape(channels, out_h, 2, out_w, 2).max(axis=(2, 4)).astype(np.int8)


def infer_i8(state, input_i8):
    debug = {"input": np.asarray(input_i8, dtype=np.int8)}

    conv1_acc = conv2d_acc_i32(debug["input"], state["conv1"]["weight_q"])
    conv1_out = requantize_i8(
        conv1_acc,
        state["conv1"]["bias_q"],
        state["conv1"]["requant_multiplier"],
        state["conv1"]["requant_shift"],
        relu=True,
    )
    pool1 = maxpool2d_i8(conv1_out)

    conv2_acc = conv2d_acc_i32(pool1, state["conv2"]["weight_q"])
    conv2_out = requantize_i8(
        conv2_acc,
        state["conv2"]["bias_q"],
        state["conv2"]["requant_multiplier"],
        state["conv2"]["requant_shift"],
        relu=True,
    )
    pool2 = maxpool2d_i8(conv2_out)

    flat = pool2.reshape(-1)
    fc1_acc = fc_acc_i32(flat, state["fc1"]["weight_q"])
    fc1_out = requantize_i8(
        fc1_acc,
        state["fc1"]["bias_q"],
        state["fc1"]["requant_multiplier"],
        state["fc1"]["requant_shift"],
        relu=True,
    )

    fc2_acc = fc_acc_i32(fc1_out, state["fc2"]["weight_q"])
    logits = requantize_i8(
        fc2_acc,
        state["fc2"]["bias_q"],
        state["fc2"]["requant_multiplier"],
        state["fc2"]["requant_shift"],
        relu=False,
    )

    debug.update({
        "layer0_acc": conv1_acc,
        "layer0_out": conv1_out,
        "layer1_acc": conv2_acc,
        "layer1_out": conv2_out,
        "layer2_acc": fc1_acc,
        "layer2_out": fc1_out,
        "layer3_acc": fc2_acc,
        "final_logits": logits,
    })
    return int(np.argmax(logits)), debug


def write_debug_hex_files(export_dir, debug):
    export_dir = Path(export_dir)
    write_hex_file(export_dir / DEBUG_FILES["input"], debug["input"], 8)
    write_hex_file(export_dir / DEBUG_FILES["layer0_acc"], debug["layer0_acc"], 32)
    write_hex_file(export_dir / DEBUG_FILES["layer0_out"], debug["layer0_out"], 8)
    write_hex_file(export_dir / DEBUG_FILES["layer1_acc"], debug["layer1_acc"], 32)
    write_hex_file(export_dir / DEBUG_FILES["layer1_out"], debug["layer1_out"], 8)
    write_hex_file(export_dir / DEBUG_FILES["layer2_acc"], debug["layer2_acc"], 32)
    write_hex_file(export_dir / DEBUG_FILES["layer2_out"], debug["layer2_out"], 8)
    write_hex_file(export_dir / DEBUG_FILES["layer3_acc"], debug["layer3_acc"], 32)
    write_hex_file(export_dir / DEBUG_FILES["final_logits"], debug["final_logits"], 8)


def main():
    parser = argparse.ArgumentParser(description="Integer-only signed symmetric INT8 inference for small_cnn_4996_params")
    parser.add_argument("--export-dir", default=".", help="Directory containing exported text files")
    parser.add_argument("--input-hex", default="input_image_i8.hex", help="Signed INT8 input hex file")
    parser.add_argument("--input-npy", default=None, help="Optional float input .npy file shaped 1x28x28 or 28x28")
    parser.add_argument("--dump-debug", action="store_true", help="Rewrite layer debug hex files")
    args = parser.parse_args()

    export_dir = Path(args.export_dir)
    state = load_exported_state(export_dir)
    qparams = load_qparams(export_dir)

    if args.input_npy:
        if "activation_scale.input" not in qparams:
            raise ValueError("input .npy quantization requires activation_scale.input in qparams")
        image = np.load(args.input_npy)
        if image.shape == (28, 28):
            image = image.reshape(1, 28, 28)
        input_i8 = symmetric_quantize_float(image, float(qparams["activation_scale.input"]))
    else:
        input_values = read_hex_file(export_dir / args.input_hex, 8)
        input_i8 = np.asarray(input_values, dtype=np.int8).reshape(1, 28, 28)

    prediction, debug = infer_i8(state, input_i8)
    print(f"prediction={prediction}")
    print("final_logits_i8=" + " ".join(str(int(value)) for value in debug["final_logits"].reshape(-1)))

    if args.dump_debug:
        write_debug_hex_files(export_dir, debug)


if __name__ == "__main__":
    main()
'''


def write_reference_infer_script(path=SYM_REFERENCE_PY):
    Path(path).write_text(REFERENCE_INFER_INT_SOURCE.lstrip())
    print(f"Wrote integer-only reference script: {path}")


write_reference_infer_script()


In [ ]:
def symmetric_quantize_array(array, scale):
    q = np.rint(np.asarray(array, dtype=np.float64) / float(scale))
    return np.clip(q, -128, 127).astype(np.int8)


def rtl_round_shift_array(values, shift):
    shift = int(np.asarray(shift).reshape(-1)[0])
    values = np.asarray(values, dtype=np.int64)
    if shift == 0:
        return values
    offset = np.int64(1 << (shift - 1))
    return np.where(values >= 0, (values + offset) >> shift, (values - offset) >> shift)


def conv2d_acc_i32(input_i8, weight_i8):
    x = np.asarray(input_i8, dtype=np.int32)
    w = np.asarray(weight_i8, dtype=np.int32)
    kh, kw = w.shape[2], w.shape[3]
    windows = np.lib.stride_tricks.sliding_window_view(x, (kh, kw), axis=(1, 2))
    return np.tensordot(w, windows, axes=([1, 2, 3], [0, 3, 4])).astype(np.int32)


def fc_acc_i32(input_i8, weight_i8):
    return (np.asarray(weight_i8, dtype=np.int32) @ np.asarray(input_i8, dtype=np.int32)).astype(np.int32)


def requantize_i8(acc_i32, bias_i32, multiplier_i32, shift_u6, relu=False):
    acc = np.asarray(acc_i32, dtype=np.int64)
    bias = np.asarray(bias_i32, dtype=np.int64).reshape((-1,) + (1,) * (acc.ndim - 1))
    mult = np.asarray(multiplier_i32, dtype=np.int64).reshape((-1,) + (1,) * (acc.ndim - 1))
    tmp = acc + bias
    scaled = rtl_round_shift_array(tmp * mult, shift_u6)
    clipped = np.clip(scaled, -128, 127)
    if relu:
        clipped = np.maximum(clipped, 0)
    return clipped.astype(np.int8)


def maxpool2d_i8(input_i8):
    x = np.asarray(input_i8, dtype=np.int8)
    channels, height, width = x.shape
    out_h, out_w = height // 2, width // 2
    x = x[:, : out_h * 2, : out_w * 2]
    return x.reshape(channels, out_h, 2, out_w, 2).max(axis=(2, 4)).astype(np.int8)


def run_symmetric_integer_inference(export_state, image):
    layers = export_state["layers"]
    image_np = image.detach().cpu().numpy() if torch.is_tensor(image) else np.asarray(image)
    if image_np.shape == (28, 28):
        image_np = image_np.reshape(1, 28, 28)

    input_i8 = symmetric_quantize_array(image_np, export_state["activation_scales"]["input"])
    debug = {"input": input_i8}

    conv1_acc = conv2d_acc_i32(input_i8, layers["conv1"]["weight_q"])
    conv1_out = requantize_i8(
        conv1_acc,
        layers["conv1"]["bias_q"],
        layers["conv1"]["requant_multiplier"],
        layers["conv1"]["requant_shift"],
        relu=True,
    )
    pool1 = maxpool2d_i8(conv1_out)

    conv2_acc = conv2d_acc_i32(pool1, layers["conv2"]["weight_q"])
    conv2_out = requantize_i8(
        conv2_acc,
        layers["conv2"]["bias_q"],
        layers["conv2"]["requant_multiplier"],
        layers["conv2"]["requant_shift"],
        relu=True,
    )
    pool2 = maxpool2d_i8(conv2_out)

    fc1_acc = fc_acc_i32(pool2.reshape(-1), layers["fc1"]["weight_q"])
    fc1_out = requantize_i8(
        fc1_acc,
        layers["fc1"]["bias_q"],
        layers["fc1"]["requant_multiplier"],
        layers["fc1"]["requant_shift"],
        relu=True,
    )

    fc2_acc = fc_acc_i32(fc1_out, layers["fc2"]["weight_q"])
    logits = requantize_i8(
        fc2_acc,
        layers["fc2"]["bias_q"],
        layers["fc2"]["requant_multiplier"],
        layers["fc2"]["requant_shift"],
        relu=False,
    )

    debug.update({
        "layer0_acc": conv1_acc,
        "layer0_out": conv1_out,
        "layer1_acc": conv2_acc,
        "layer1_out": conv2_out,
        "layer2_acc": fc1_acc,
        "layer2_out": fc1_out,
        "layer3_acc": fc2_acc,
        "final_logits": logits,
    })
    return {"prediction": int(np.argmax(logits)), "debug": debug}


def signed_to_hex(value, bits):
    return f"{int(value) & ((1 << bits) - 1):0{bits // 4}x}"


def write_hex_file(path, values, bits):
    flat = np.asarray(values).reshape(-1)
    Path(path).write_text("".join(f"{signed_to_hex(value, bits)}\n" for value in flat))


def write_symmetric_debug_hex_files(debug, files=SYM_DEBUG_HEX_FILES):
    write_hex_file(files["input"], debug["input"], 8)
    write_hex_file(files["layer0_acc"], debug["layer0_acc"], 32)
    write_hex_file(files["layer0_out"], debug["layer0_out"], 8)
    write_hex_file(files["layer1_acc"], debug["layer1_acc"], 32)
    write_hex_file(files["layer1_out"], debug["layer1_out"], 8)
    write_hex_file(files["layer2_acc"], debug["layer2_acc"], 32)
    write_hex_file(files["layer2_out"], debug["layer2_out"], 8)
    write_hex_file(files["layer3_acc"], debug["layer3_acc"], 32)
    write_hex_file(files["final_logits"], debug["final_logits"], 8)
    print("Wrote RTL debug hex files:")
    for output_path in files.values():
        print(f"  {output_path}")


def run_symmetric_integer_sample(export_state, dataset, sample_index=INFERENCE_SAMPLE_INDEX):
    image, label = dataset[sample_index]
    result = run_symmetric_integer_inference(export_state, image)
    result.update({
        "sample_index": int(sample_index),
        "label": int(label),
        "correct": result["prediction"] == int(label),
    })
    write_symmetric_debug_hex_files(result["debug"])
    print(f"Signed symmetric INT8 sample {sample_index}: pred={result['prediction']}, label={result['label']}, correct={result['correct']}")
    return result


def evaluate_symmetric_integer_accuracy(export_state, dataset, max_images=SYM_ACCURACY_MAX_IMAGES):
    total = len(dataset) if max_images is None else min(int(max_images), len(dataset))
    correct = 0
    start = time.perf_counter()
    for index in range(total):
        image, label = dataset[index]
        result = run_symmetric_integer_inference(export_state, image)
        correct += int(result["prediction"] == int(label))
    elapsed = time.perf_counter() - start
    accuracy = 100.0 * correct / total if total else 0.0
    ms_per_image = 1000.0 * elapsed / total if total else 0.0
    print(f"Signed symmetric INT8 integer accuracy: {correct}/{total} ({accuracy:.2f}%)")
    print(f"Signed symmetric INT8 integer inference: {ms_per_image:.6f} ms/image")
    return {
        "correct": correct,
        "total": total,
        "accuracy": accuracy,
        "ms_per_image": ms_per_image,
    }


sym_inference_result = run_symmetric_integer_sample(
    sym_export_state,
    test_loader.dataset,
    sample_index=INFERENCE_SAMPLE_INDEX,
)
sym_accuracy_result = evaluate_symmetric_integer_accuracy(sym_export_state, test_loader.dataset)


In [ ]:
print("Summary")
print(f"Float accuracy: {float_accuracy:.2f}%")
print(f"QAT accuracy before convert: {qat_accuracy:.2f}%")
print(f"PyTorch affine INT8 accuracy: {int8_accuracy:.2f}%")
print(f"Float inference: {float_ms:.6f} ms/image")
print(f"PyTorch affine INT8 inference: {int8_ms:.6f} ms/image")
if "sym_accuracy_result" in globals():
    print(f"Signed symmetric INT8 integer accuracy: {sym_accuracy_result['accuracy']:.2f}%")
    print(f"Signed symmetric INT8 integer inference: {sym_accuracy_result['ms_per_image']:.6f} ms/image")
if "sym_export_info" in globals():
    print(f"SYM weights: {sym_export_info['weights_path']} ({sym_export_info['weight_count']} values)")
    print(f"SYM biases: {sym_export_info['biases_path']} ({sym_export_info['bias_count']} values)")
    print(f"SYM requant multipliers: {sym_export_info['requant_mult_path']} ({sym_export_info['requant_count']} values)")
    print(f"SYM requant shifts: {sym_export_info['requant_shift_path']} ({sym_export_info['requant_count']} values)")
    print(f"SYM qparams: {sym_export_info['qparams_path']}")
    print(f"Integer reference script: {SYM_REFERENCE_PY}")
if "sym_inference_result" in globals():
    print(
        f"SYM sample {sym_inference_result['sample_index']}: "
        f"pred={sym_inference_result['prediction']}, "
        f"label={sym_inference_result['label']}, "
        f"correct={sym_inference_result['correct']}"
    )
